## Overview

- how the logging system works
- customizing the logging system
- storing and reading logs from aws
- monitoring your dags with Elasticsearch and Kibana
- what are the metrics available with Airflow
- monitoring your Airflow instance with the TIG (Telegraf, Influx, Grafana) stack
- Maintenance DAGs for Airflow in prod
- and more

## Logging Basics

- based on the logging module for Python
- written into files
- log level: INFO, ERROR, DEBUG, WARNING, CRITICAL
- formatters - sets how the output of logs should look like
- handlers for outputs: FileHandler, StreamHandler, NullHandler

Example:
```python
def setup_logging(filename):
    root = logging.getLogger()
    handler = logging.FileHandler(filename)
    formatter = logging.Formatter(settings.SIMPLE_LOG_FORMAT)
    handler.setFormatter(formatter)
    root.addHandler(handler)
    root.setLevel(settings.LOGGING_LEVEL)

    return handler.strea
```
Where the logs are stored? (Depends on the handler)
- File
- Stream
- S3
- ES
- GCS

For external handlers such as S3 and ElasticSearch, a connection. The parameters REMOTE_LOG_CONN_ID must be set with the connectio to the system and REMOTE_LOGGING must be set to true.

## Elasticsearch

Elasticsearch is an open source search engine providing a distributed full-text search engine with an HTTP web interface and schema-free JSON documents. 
It allows you to store, search and analyze large volumes of data.
It is well suited for collecting and aggregating log data at scale to look for trends,statistics, summarizations and more in near real time.

There are two concepts to know in Elasticsearch. 

The first one is the document. A document can be seen as a row in a relational database and it corresponds to your data stored in JSON format into Elasticsearch.
So basically, a document could be the following JSON data coming from a log file with different fields such as Id, Name, and Loglevel as well as their associated values.

Next, these documents are stored into an Index which is a collection of documents. 
You can think of an index as a database from a relational database perspective. 
For scalability reasons, it’s a best practice to create a new index for each day when you are dealing with log events in Elasticsearch.
By doing this, you will be able to request your logs according to a given date or a range of dates, avoiding requesting all your data which would be very slow and non optimized.

An index can have one or more mapping types that are used to divide documents into logical groups inside the same index.
A mapping defines how a document is indexed and how its fields, are indexed and stored.
You can think of a mapping as a table in a relational database.

Once your documents are mapped and stored into an index, you can start requesting them through the REST API given by Elasticsearch.

#### Kibana
Kibana is an open source data visualization tool used for log analytics, application monitoring and more. It gives you a powerful and easy way to make dashboards on top of your data stored in Elasticsearch.

#### Logstash
Logstash is a server side data processing pipeline that can ingest data from multiple sources simultaneously, transform it and ship it to the output you want such as Elasticsearch, Statsd, Kafka and so on.
In other words, it allows you to ingest data of different shapes and sources, then parse each event in order to build a common format to finally send the processed data to one or multiple systems.
If you want to apply aggregations, filters or transformations to your log events before forwarding them, then Logstach could be usefule to you.

Together they are know as the ELK stack.

#### Filebeat
Filebeat monitors the log files or locations that you specify, collect log events, and forwards them to either Elasticsearch or Logstash for indexing. Like logstash, you have to define an input and an output but the transformations you can make on your data are very limited.
Nonetheless, Filebeat has a very small CPU and memory footprint and that’s why it is usually installed on the same node where the logs are in order to forward them to Logstash which is running on another node since it consumes a lot more resources.

## Metrics

#### StatsD
- Airflow sends metrics to StatsD
- Daemon to aggregate and summarize app metrics
- extremely fast via UDP and tiny resource footprint
- forwards metrics to other apps (push to apps such as ElasticSearch)


All metrics sent by Airflow are based on:
- counters
- gauges
- timers

See: https://airflow.apache.org/docs/stable/metrics.html

#### TIG Stack
Telegraf
- an open source agent for collecting, processing and aggregating metrics.Once the data are collected from various kinds of inputs, they are pushed to different outputs such as InfluxDB, Elasticsearch, Syslog and so on. It is very lightweight and works with a system of plugins that you can add or remove from each step your metrics go through.

InfluxDB
- InfluxDB is an open source time series database built from the ground up to handle high write and query loads. The purpose of InfluxDB is to be used as a backing store for any use case involving large amounts of time stamped data.

Grafana
- Grafana is an open source data visualization and monitoring application. It supports many databases and gives the ability to create complex and beautiful dashboards mixing different data sources at one place. Grafana is also widely used to set alerts according to the metrics and thresholds you have.

There are the inputs, processors, aggregators and outputs. In our case, we are going to use the StatsD input plugin which runs the StatsD daemon to be able to receive metrics from Airflow. Then, we will send the metrics to InfluxDB. Finally, once the metrics are stored into InfluxDB, we will visualize and monitor them through Grafana.

To sum up, the TIG stack is a really common stack for monitoring systems since each component works mostly with each other.

Note:
- The package statsd must be installed along with the other packages such as crypto, postgres and so on. Otherwise Airflow won’t be able to send the metrics.
- The parameter statsd_allow_list allows you to filter which metrics you want to send by specifying a list of prefixes corresponding to the metrics of Airflow.